<a href="https://colab.research.google.com/github/NathiJonas/Agentic-AI-Andela-Work/blob/main/pomodoro_deep_research_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pomodoro Deep Research System

A Pomodoro timer fused with the Deep Research agentic pipeline.

**Timer:** Focus (25 min) → Short Break (5 min) → Long Break (15 min, every 4 sessions)  
**Research pipeline:** Safety → Clarify → Plan → Search (parallel) → Sufficiency → Write → Evaluate  

**Features:**
- Gradio UI with three tabs: Timer, Research, Report
- Queue a research topic during Focus; it auto-launches when break starts
- Live status log streams pipeline progress in real time
- Web search via Anthropic built-in `web_search_20250305` tool
- Input guardrail blocks harmful queries; output guardrail checks word count

In [ ]:

%pip install anthropic gradio==5.23.0 python-dotenv pydantic --quiet

In [ ]:
import asyncio
import json
import os
import sys
import tempfile
import pathlib
import subprocess

import anthropic
import gradio as gr
from dotenv import load_dotenv

load_dotenv()

import os
os.environ["ANTHROPIC_API_KEY"] = os.environ.get("ANTHROPIC_API_KEY", "")

MODEL        = "claude-opus-4-5"
GUARD_MODEL  = "claude-haiku-4-5-20251001"
HAIKU_MODEL  = "claude-haiku-4-5-20251001"
MAX_TOKENS   = 2048
MAX_USER_MSG_LENGTH = 2000
MAX_TURNS    = 15

api_key = os.environ.get("ANTHROPIC_API_KEY")
print(f"Key loaded: {api_key[:10]}..." if api_key else "ERROR: No API key found!")

client = anthropic.AsyncAnthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", ""))
print("Client OK")

print("Imports OK.")

test_client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", ""))


In [ ]:
class ClarifyingQuestions(BaseModel):
    questions:       list[str] = Field(description="Three clarifying questions to focus the research.")
    context_summary: str       = Field(description="One-sentence summary of the query.")

class WebSearchItem(BaseModel):
    query:    str = Field(description="The search term.")
    reason:   str = Field(description="Why this search is relevant.")
    priority: int = Field(description="Priority 1 (critical) to 3 (nice-to-have).", ge=1, le=3)

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="Ordered list of web searches.")

class ResearchSufficiencyCheck(BaseModel):
    is_sufficient:      bool      = Field(description="True if current results are enough for a thorough report.")
    missing_aspects:    list[str] = Field(description="Topics still not covered.")
    additional_queries: list[str] = Field(description="New queries to fill the gaps.")
    reasoning:          str       = Field(description="Brief explanation.")

class ReportData(BaseModel):
    short_summary:       str       = Field(description="2-3 sentence executive summary.")
    markdown_report:     str       = Field(description="Full detailed report in Markdown (min 1000 words).")
    follow_up_questions: list[str] = Field(description="Suggested directions for further research.")

class ReportEvaluation(BaseModel):
    is_approved:             bool      = Field(description="True if the report meets quality standards.")
    score:                   float     = Field(description="Quality score 0-10.", ge=0, le=10)
    strengths:               list[str] = Field(description="What the report does well.")
    weaknesses:              list[str] = Field(description="Areas that fall short.")
    improvement_suggestions: list[str] = Field(description="Actionable suggestions.")

class SafetyResult(BaseModel):
    is_safe: bool
    reason:  str

print("Schemas OK.")

In [ ]:
async def call_claude(
    system: str,
    user: str,
    max_tokens: int = 2048,
    temperature: float = 0.3,
) -> str:
    """Single-turn Claude call. Returns assistant text."""
    response = await client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        temperature=temperature,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    return "".join(b.text for b in response.content if hasattr(b, "text"))


async def call_claude_json(
    system: str,
    user: str,
    schema: type,
    max_tokens: int = 2048,
) -> object:
    """Call Claude and parse JSON response validated against a Pydantic schema."""
    sys_with_json = (
        system
        + "\n\nRespond with valid JSON only matching this schema (no markdown fences, no preamble): "
        + json.dumps(schema.model_json_schema())
    )
    text = await call_claude(sys_with_json, user, max_tokens=max_tokens, temperature=0.2)
    clean = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return schema.model_validate_json(clean)




async def call_claude_haiku(
    system: str,
    user: str,
    schema: type,
    max_tokens: int = 512,
) -> object:
    """Lightweight structural call using Haiku to avoid Opus rate limits."""
    sys_with_json = (
        system
        + "\n\nRespond with valid JSON only matching this schema (no markdown fences, no preamble): "
        + json.dumps(schema.model_json_schema())
    )
    response = await client.messages.create(
        model=HAIKU_MODEL,
        max_tokens=max_tokens,
        temperature=0.2,
        system=sys_with_json,
        messages=[{"role": "user", "content": user}],
    )
    text = "".join(b.text for b in response.content if hasattr(b, "text"))
    clean = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return schema.model_validate_json(clean)


WEB_SEARCH_TOOL = {"type": "web_search_20250305", "name": "web_search"}

async def web_search(query: str) -> str:
    """Search the web using Claude's built-in web_search tool."""
    try:
        response = await client.messages.create(
            model=MODEL,
            max_tokens=1024,
            temperature=0.1,
            tools=[WEB_SEARCH_TOOL],
            system=(
                "You are a research assistant. Use the web_search tool to find current information, "
                "then summarize the top results in 2-3 concise paragraphs (note-taking style, no filler). "
                "Always call the tool before summarizing."
            ),
            messages=[{"role": "user", "content": f"Search for: {query}"}],
        )
        return "".join(b.text for b in response.content if hasattr(b, "text")) or f"No results for: {query}"
    except Exception as exc:
        return f"Search error for '{query}': {exc}"


async def run_parallel_searches(queries: list[str]) -> str:
    """Execute web searches sequentially with throttling to respect rate limits."""
    async def _one(q: str) -> str:
        try:
            result = await web_search(q)
            return f"### {q}\n{result}\n"
        except Exception as exc:
            return f"### {q}\nFailed: {exc}\n"

    results = []
    for i, q in enumerate(queries[:6]):
        if i > 0:
            await asyncio.sleep(2)
        results.append(await _one(q))
    return "\n---\n".join(results)

print("Helpers OK.")

In [ ]:
async def clarifier_agent(query: str) -> ClarifyingQuestions:
    return await call_claude_haiku(
        "Given a research query, generate exactly three clarifying questions that sharpen the scope, "
        "and provide a one-sentence summary.",
        query,
        ClarifyingQuestions,
    )


async def planner_agent(query: str, clarifications: str) -> WebSearchPlan:
    return await call_claude_haiku(
        "Given a query and clarifications, produce 5-8 prioritised web search terms "
        "(priority 1=critical, 2=important, 3=nice-to-have).",
        f"Query: {query}\n\n{clarifications}",
        WebSearchPlan,
    )


async def sufficiency_agent(query: str, results: str) -> ResearchSufficiencyCheck:

    snippet = results[:2000]
    return await call_claude_haiku(
        "Given the query and a snippet of search results, decide whether evidence is sufficient "
        "for a thorough report. Suggest up to 3 additional queries if gaps remain.",
        f"Query: {query}\n\nSearch results (snippet):\n{snippet}",
        ResearchSufficiencyCheck,
        max_tokens=512,
    )


async def writer_agent(query: str, results: str) -> ReportData:
    return await call_claude_json(
        "Given the query and search results, produce a detailed Markdown report (1000-2000 words): "
        "executive summary, background, key findings, analysis, conclusions, follow-up questions. "
        "Cite sources inline where available.",
        f"Query: {query}\n\nSearch results:\n{results[:8000]}",
        ReportData,
        max_tokens=4096,
    )


async def evaluator_agent(query: str, report: str) -> ReportEvaluation:
    return await call_claude_haiku(
        "Score the report 0-10. Approve (score >= 7) only if it fully addresses the query, "
        "is factually grounded, well structured, and at least 800 words. "
        "For lower scores provide actionable improvement suggestions.",
        f"Query: {query}\n\nReport (first 3000 chars):\n{report[:3000]}",
        ReportEvaluation,
        max_tokens=512,
    )


async def safety_agent(query: str) -> SafetyResult:
    return await call_claude_haiku(
        "Decide whether the research query is safe. Flag UNSAFE for: instructions to cause harm, "
        "illegal activity, synthesis of dangerous substances, or harassment of private individuals. "
        "Legitimate research on sensitive topics is SAFE.",
        query,
        SafetyResult,
        max_tokens=128,
    )

print("Agents OK.")

In [ ]:
async def run_research_pipeline(query: str):
    """
    Full agentic research pipeline.
    Yields (status_log, report_markdown) tuples for the Gradio UI.
    """
    if not query.strip():
        yield "Please enter a research query.", ""
        return

    status, report_md = "", ""

    # ── Input guardrail
    status += "Checking query safety...\n"
    yield status, report_md
    try:
        safety = await safety_agent(query.strip())
        if not safety.is_safe:
            status += f"BLOCKED: {safety.reason}\n"
            yield status, report_md
            return
        status += "Query approved.\n\n"
        yield status, report_md
    except Exception as exc:
        status += f"Safety check unavailable ({exc}), proceeding.\n\n"
        yield status, report_md

    # ── Stage 1: Clarify ──────────────────────────────────────────────────────
    status += "[Stage 1] Analysing research scope...\n"
    yield status, report_md
    clarifications = ""
    try:
        cr = await clarifier_agent(query.strip())
        qs = cr.questions[:3]
        status += f"{cr.context_summary}\n\nScope questions:\n" + "\n".join(f"  - {q}" for q in qs) + "\n\n"
        yield status, report_md
        clarifications = "Scope questions:\n" + "\n".join(f"- {q}" for q in qs)
    except Exception as exc:
        status += f"Warning: scope analysis skipped ({exc})\n\n"
        yield status, report_md

    # ── Stage 2: Plan ─────────────────────────────────────────────────────────
    status += "[Stage 2] Planning search strategy...\n"
    yield status, report_md
    search_queries: list[str] = []
    try:
        plan = await planner_agent(query, clarifications)
        searches_sorted = sorted(plan.searches, key=lambda s: s.priority)
        critical = [s for s in searches_sorted if s.priority == 1]
        status += f"Planned {len(searches_sorted)} searches ({len(critical)} critical).\n\n"
        yield status, report_md
        search_queries = [s.query for s in searches_sorted if s.priority <= 2][:6]
    except Exception as exc:
        status += f"Planner failed ({exc}), using fallback queries.\n\n"
        search_queries = [query, query + " latest research", query + " analysis"]
        yield status, report_md

    # ── Stage 3: Search ───────────────────────────────────────────────────────
    status += f"[Stage 3] Running {len(search_queries)} parallel searches...\n"
    for q in search_queries:
        status += f"  -> {q}\n"
    yield status, report_md

    all_results = await run_parallel_searches(search_queries)
    status += "Searches complete.\n\n"
    yield status, report_md

    # ── Stage 4: Sufficiency (with retries) ───────────────────────────────────
    for retry in range(MAX_SEARCH_RETRIES + 1):
        status += f"[Stage 4] Checking research sufficiency (pass {retry + 1})...\n"
        yield status, report_md
        try:
            suf = await sufficiency_agent(query, all_results)
            if suf.is_sufficient or retry >= MAX_SEARCH_RETRIES:
                msg = f"Coverage sufficient: {suf.reasoning[:120]}" if suf.is_sufficient else "Coverage thin but max retries reached."
                status += msg + "\n\n"
                yield status, report_md
                break
            else:
                missing = ", ".join(suf.missing_aspects[:3])
                status += f"Gaps found: {missing}\nRunning {len(suf.additional_queries[:3])} extra searches...\n"
                yield status, report_md
                extra = await run_parallel_searches(suf.additional_queries[:3])
                all_results += "\n---\n" + extra
                status += "Extra searches complete.\n\n"
                yield status, report_md
        except Exception as exc:
            status += f"Sufficiency check failed (Error: {str(exc)[:120]}), proceeding.\n\n"
            yield status, report_md
            break

    # ── Stage 5: Write (with retry) ───────────────────────────────────────────
    final_report_md = ""
    for retry in range(MAX_REPORT_RETRIES + 1):
        status += f"[Stage 5] Writing report (attempt {retry + 1})...\n"
        yield status, report_md
        try:
            wr = await writer_agent(query, all_results)
            final_report_md = wr.markdown_report
            wc = len(final_report_md.split())
            status += f"Draft ready ({wc} words).\n\n"
            report_md = final_report_md
            yield status, report_md
        except Exception as exc:
            status += f"Writer error ({exc}).\n\n"
            yield status, report_md
            break

        # ── Stage 6: Evaluate ─────────────────────────────────────────────────
        status += "[Stage 6] Evaluating report quality...\n"
        yield status, report_md
        try:
            ev = await evaluator_agent(query, final_report_md)
            status += f"Score: {ev.score:.1f}/10 -- {'APPROVED' if ev.is_approved else 'needs revision'}\n"
            if ev.strengths:
                status += f"  [+] {ev.strengths[0]}\n"
            if not ev.is_approved and ev.improvement_suggestions:
                status += f"  [!] {ev.improvement_suggestions[0]}\n"
            status += "\n"
            yield status, report_md
            if ev.is_approved or retry >= MAX_REPORT_RETRIES:
                break
        except Exception as exc:
            status += f"Evaluation skipped ({exc}).\n\n"
            yield status, report_md
            break

    # ── Output guardrail
    wc = len(final_report_md.split())
    if final_report_md and wc < 200:
        status += f"Warning: report too short ({wc} words). Consider re-running.\n"
        yield status, report_md

    status += "Pipeline complete!\n"
    yield status, report_md

print("Pipeline OK.")

In [ ]:
timer_state = {
    "mode":          "focus",
    "seconds_left":  25 * 60,
    "running":       False,
    "sessions":      0,
    "queued_query":  "",
    "auto_launch":   False,
    "thread":        None,
    "last_tick":     0.0,
}


def timer_tick():
    """Background thread: decrements seconds_left once per second."""
    while True:
        time.sleep(1)
        if timer_state["running"] and timer_state["seconds_left"] > 0:
            timer_state["seconds_left"] -= 1
            timer_state["last_tick"] = time.time()
        elif timer_state["running"] and timer_state["seconds_left"] <= 0:
            timer_state["running"] = False
            _handle_timer_end()


def _handle_timer_end():
    """Advance to the next Pomodoro phase when a timer expires."""
    mode = timer_state["mode"]
    if mode == "focus":
        timer_state["sessions"] += 1
        next_mode = "long_break" if timer_state["sessions"] % 4 == 0 else "short_break"
        _switch_mode(next_mode, auto_start=True)
        if timer_state["auto_launch"] and timer_state["queued_query"].strip():

            _pending_auto_research[0] = timer_state["queued_query"]
    else:
        _switch_mode("focus", auto_start=False)


def _switch_mode(mode: str, auto_start: bool = False):
    timer_state["mode"] = mode
    timer_state["seconds_left"] = MODES[mode]["minutes"] * 60
    timer_state["running"] = auto_start



_pending_auto_research = [""]

_t = threading.Thread(target=timer_tick, daemon=True)
_t.start()

print("Timer thread started.")

In [ ]:
def fmt_time(secs: int) -> str:
    m, s = divmod(max(secs, 0), 60)
    return f"{m:02d}:{s:02d}"


def build_timer_display():
    """Return (clock_str, mode_label, sessions_str, progress_pct)."""
    mode    = timer_state["mode"]
    secs    = timer_state["seconds_left"]
    total   = MODES[mode]["minutes"] * 60
    pct     = max(0.0, min(1.0, 1.0 - secs / total)) * 100
    dots    = ("[X] " * (timer_state["sessions"] % 4)) + ("[ ] " * (4 - timer_state["sessions"] % 4))
    return (
        fmt_time(secs),
        MODES[mode]["label"],
        f"Sessions completed: {timer_state['sessions']}  |  {dots.strip()}",
        pct,
    )

print("Display helper OK.")

In [ ]:
def cb_start_pause():
    timer_state["running"] = not timer_state["running"]
    label = "Pause" if timer_state["running"] else "Start"
    clk, mode_lbl, sessions_str, pct = build_timer_display()
    return clk, mode_lbl, sessions_str, pct, gr.update(value=label)


def cb_reset():
    timer_state["running"] = False
    _switch_mode(timer_state["mode"])
    clk, mode_lbl, sessions_str, pct = build_timer_display()
    return clk, mode_lbl, sessions_str, pct, gr.update(value="Start")


def cb_set_mode(mode: str):
    timer_state["running"] = False
    _switch_mode(mode)
    clk, mode_lbl, sessions_str, pct = build_timer_display()
    return clk, mode_lbl, sessions_str, pct, gr.update(value="Start")


def cb_tick():
    """Called by Gradio every() ticker to refresh the timer display."""
    clk, mode_lbl, sessions_str, pct = build_timer_display()
    return clk, mode_lbl, sessions_str, pct


def cb_queue(query: str, auto: bool):
    timer_state["queued_query"] = query.strip()
    timer_state["auto_launch"] = auto
    return gr.update(value=f"Queued: {query.strip()}") if query.strip() else gr.update(value="")


# --- Research callbacks ---

def cb_run_research(query: str):
    """Sync wrapper: runs the async pipeline via asyncio.run in a new event loop."""
    if not query.strip():
        yield "Please enter a research query.", ""
        return

    async def _run():
        async for status, report in run_research_pipeline(query):
            yield status, report


    loop = asyncio.new_event_loop()
    try:
        gen = _run()
        while True:
            try:
                status, report = loop.run_until_complete(gen.__anext__())
                yield status, report
            except StopAsyncIteration:
                break
    finally:
        loop.close()

print("Callbacks OK.")

In [ ]:

CUSTOM_CSS = """
#clock     { font-size: 4rem; font-weight: 700; text-align: center; letter-spacing: 4px; font-family: monospace; }
#mode_lbl  { font-size: 1.1rem; text-align: center; letter-spacing: 3px; text-transform: uppercase; color: #888; }
#sessions  { text-align: center; font-family: monospace; color: #555; }
.big-btn   { font-size: 1.1rem !important; font-weight: 700 !important; }
.log-box textarea { font-family: 'Courier New', monospace !important; font-size: 0.82rem !important; }
"""

with gr.Blocks(title="Pomodoro Deep Research", css=CUSTOM_CSS) as demo:

    gr.Markdown("# Pomodoro Deep Research System")
    gr.Markdown(
        "Focus with the Pomodoro timer; queue a research topic to launch automatically "
        "at the start of each break. Pipeline: "
        "**Safety -> Clarify -> Plan -> Search (parallel) -> Sufficiency -> Write -> Evaluate**"
    )

    with gr.Tab("Timer"):

        with gr.Row():
            with gr.Column(scale=1):
                clock_display   = gr.Textbox(value="25:00", label="", elem_id="clock", interactive=False)
                mode_display    = gr.Textbox(value="Focus", label="", elem_id="mode_lbl", interactive=False)
                sessions_display = gr.Textbox(value="Sessions completed: 0  |  [ ] [ ] [ ] [ ]",
                                              label="", elem_id="sessions", interactive=False)
                progress_bar    = gr.Slider(value=0, minimum=0, maximum=100,
                                            label="Progress", interactive=False)

            with gr.Column(scale=1):
                with gr.Row():
                    btn_mode_focus  = gr.Button("Focus (25m)",       elem_classes="big-btn")
                    btn_mode_short  = gr.Button("Short Break (5m)",  elem_classes="big-btn")
                    btn_mode_long   = gr.Button("Long Break (15m)",  elem_classes="big-btn")

                with gr.Row():
                    btn_start_pause = gr.Button("Start", variant="primary", elem_classes="big-btn")
                    btn_reset       = gr.Button("Reset",               elem_classes="big-btn")

                gr.Markdown("---")
                gr.Markdown("### Break Research Queue")
                gr.Markdown(
                    "Type a research topic below and click **Queue**. "
                    "Enable auto-launch to start research automatically when your break begins."
                )
                queue_input  = gr.Textbox(
                    label="Topic to research on break",
                    placeholder="e.g. Latest advances in fusion energy",
                )
                auto_launch  = gr.Checkbox(label="Auto-launch research at break start", value=False)
                btn_queue    = gr.Button("Queue Topic")
                queue_status = gr.Textbox(label="", interactive=False, show_label=False)

        ticker = gr.Timer(value=1)

    with gr.Tab("Research"):

        with gr.Row():
            with gr.Column(scale=1):
                research_query  = gr.Textbox(
                    label="Research Query",
                    placeholder="What would you like to research?",
                    lines=3,
                )
                research_btn    = gr.Button("Run Research", variant="primary", elem_classes="big-btn")
                gr.Markdown(
                    "_Pipeline: Safety -> Clarify -> Plan -> Search -> "
                    "Sufficiency -> Write -> Evaluate_"
                )

            with gr.Column(scale=2):  # <-- fixed: now inside with gr.Row()
                with gr.Tab("Live Status"):
                    status_box = gr.Textbox(
                        label="",
                        lines=22,
                        interactive=False,
                        show_copy_button=True,
                        placeholder="Status updates will stream here as the agent works...",
                        elem_classes="log-box",
                    )
                with gr.Tab("Final Report"):
                    report_box = gr.Markdown("The report will appear here when ready.")


    timer_outputs = [clock_display, mode_display, sessions_display, progress_bar]

    btn_start_pause.click(
        fn=cb_start_pause,
        outputs=timer_outputs + [btn_start_pause],
    )
    btn_reset.click(
        fn=cb_reset,
        outputs=timer_outputs + [btn_start_pause],
    )
    btn_mode_focus.click(
        fn=lambda: cb_set_mode("focus"),
        outputs=timer_outputs + [btn_start_pause],
    )
    btn_mode_short.click(
        fn=lambda: cb_set_mode("short_break"),
        outputs=timer_outputs + [btn_start_pause],
    )
    btn_mode_long.click(
        fn=lambda: cb_set_mode("long_break"),
        outputs=timer_outputs + [btn_start_pause],
    )

    btn_queue.click(
        fn=cb_queue,
        inputs=[queue_input, auto_launch],
        outputs=[queue_status],
    )

    ticker.tick(
        fn=cb_tick,
        outputs=timer_outputs,
    )

    research_btn.click(
        fn=cb_run_research,
        inputs=[research_query],
        outputs=[status_box, report_box],
    )
    research_query.submit(
        fn=cb_run_research,
        inputs=[research_query],
        outputs=[status_box, report_box],
    )

demo.launch(inbrowser=True)